# Tutorial 4: Galactic Priors and Models

This tutorial explores the prior probability distributions used in brutus for modeling stars in the Milky Way context.

## Topics Covered

1. **IMF priors** for stellar masses
2. **Galactic structure priors** (thin disk, thick disk, halo)
3. **3D dust priors** with Bayestar
4. **Distance and parallax priors**
5. **Prior factorization** and combination

## Prerequisites

This tutorial requires the following brutus data files:
- `nn_c3k.h5` - Neural network for bolometric corrections
- `MIST_1.2_iso_vvcrit0.0.h5` - MIST isochrones
- `bayestar2019_v1.h5` (optional) - Bayestar dust map

If you don't have these files, run the optional download cell below.

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import tutorial utilities
from tutorial_utils import (
    set_plot_style,
    find_brutus_data_file,
    save_figure as save_fig_util,
    print_section
)

# Set plot style
set_plot_style()
plt.rcParams['figure.figsize'] = (10, 6)

# Create plots directory if needed
plots_dir = Path('plots/tutorial_04')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Create plots directory if needed
plots_dir = Path('plots/tutorial_04')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

In [ ]:
# Helper function to find data files
def find_brutus_data_file(filename):
    """Find brutus data file in common locations."""
    import os
    
    # Common search paths
    search_paths = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path.home() / '.brutus' / 'data',
        Path('/mnt/d/Dropbox/GitHub/brutus/data'),
        Path('/mnt/c/Dropbox/GitHub/brutus/data'),
    ]
    
    for base_path in search_paths:
        filepath = base_path / filename
        if filepath.exists():
            return str(filepath)
    
    # Try environment variable
    if 'BRUTUS_DATA_DIR' in os.environ:
        filepath = Path(os.environ['BRUTUS_DATA_DIR']) / filename
        if filepath.exists():
            return str(filepath)
    
    raise FileNotFoundError(f"Could not find {filename}. Please download it or set BRUTUS_DATA_DIR.")

## Section 1: IMF Priors - The Distribution of Stellar Masses

The Initial Mass Function (IMF) describes the distribution of stellar masses at birth. Brutus supports several standard IMFs.

### Key Concepts

- **Kroupa IMF**: Broken power law with 3 segments
- **Salpeter IMF**: Single power law
- **Different slopes** affect low-mass vs high-mass star counts
- **Observational implications**: M/L ratios, observable fractions

In [ ]:
from brutus.priors.stellar import logp_imf

# Create IMF comparison plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Different IMF functional forms
ax = axes[0, 0]
mass_grid = np.logspace(-1.5, 2.5, 1000)

# Define IMF parameters
imf_params = {
    'Kroupa': (1.3, 2.3, 0.5),
    'Salpeter': (1.35, 2.35, 0.5),
    'Steep': (1.5, 2.5, 0.5)
}

colors = ['blue', 'red', 'green']
linestyles = ['-', '--', ':']

for (name, params), color, ls in zip(imf_params.items(), colors, linestyles):
    lnp = logp_imf(mass_grid, alpha_low=params[0], alpha_high=params[1], mass_break=params[2])
    p = np.exp(lnp - np.max(lnp))
    ax.loglog(mass_grid, p, color=color, ls=ls, lw=2, label=name)

ax.set_xlabel('Initial Mass (M☉)')
ax.set_ylabel('ξ(M) (normalized)')
ax.set_title('IMF Functional Forms')
ax.set_xlim(0.01, 100)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Power-law slopes
ax = axes[0, 1]

mass_ranges = [(0.01, 0.08), (0.08, 0.5), (0.5, 100)]
slopes = [-0.3, -1.3, -2.3]
colors_slope = ['purple', 'blue', 'cyan']

for (m_min, m_max), slope, color in zip(mass_ranges, slopes, colors_slope):
    m_range = np.logspace(np.log10(m_min), np.log10(m_max), 100)
    y = m_range**slope
    ax.loglog(m_range, y/y[0], color=color, lw=3, alpha=0.7, label=f'α = {slope:.1f}')

ax.set_xlabel('Mass (M☉)')
ax.set_ylabel('Power Law')
ax.set_title('Kroupa IMF Slopes')
ax.set_xlim(0.01, 100)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Mass sampling histogram
ax = axes[0, 2]

# Sample from IMF using rejection sampling
def sample_imf(n, alpha_low, alpha_high, mass_break=0.5, m_min=0.08, m_max=100):
    samples = []
    while len(samples) < n:
        candidates = np.random.uniform(m_min, m_max, 1000)
        log_probs = logp_imf(candidates, alpha_low=alpha_low, alpha_high=alpha_high, mass_break=mass_break)
        probs = np.exp(log_probs - np.max(log_probs))
        u = np.random.uniform(0, 1, len(candidates))
        accepted = candidates[u < probs]
        samples.extend(accepted)
    return np.array(samples[:n])

n_samples = 10000
for name, color in [('Kroupa', 'blue'), ('Salpeter', 'red')]:
    params = imf_params[name]
    masses = sample_imf(n_samples, alpha_low=params[0], alpha_high=params[1], mass_break=params[2])
    bins = np.logspace(-1, 2, 50)
    ax.hist(masses, bins=bins, alpha=0.5, color=color, label=name, histtype='step', lw=2)

ax.set_xlabel('Sampled Mass (M☉)')
ax.set_ylabel('Number of Stars')
ax.set_title(f'Random Samples (N={n_samples})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Cumulative mass function
ax = axes[1, 0]

for name, color in zip(['Kroupa', 'Salpeter', 'Steep'], colors):
    params = imf_params[name]
    masses = sample_imf(10000, alpha_low=params[0], alpha_high=params[1], mass_break=params[2])
    masses_sorted = np.sort(masses)
    cumulative = np.arange(len(masses)) / len(masses)
    ax.semilogx(masses_sorted[::10], cumulative[::10], color=color, lw=2, label=name)

ax.set_xlabel('Mass (M☉)')
ax.set_ylabel('Cumulative Fraction')
ax.set_title('Cumulative Mass Functions')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 5: Mass-to-light ratios
ax = axes[1, 1]

imf_ml = {'Kroupa': 0.6, 'Salpeter': 1.0, 'Steep': 0.5}
bars = ax.bar(imf_ml.keys(), imf_ml.values(), color=['blue', 'red', 'green'], alpha=0.7)
ax.set_ylabel('M/L (solar units)')
ax.set_title('Mass-to-Light Ratios')
ax.grid(True, alpha=0.3, axis='y')

# Panel 6: Observable fraction
ax = axes[1, 2]

mass_limits = np.logspace(-1, 0.5, 20)

for name, color in [('Kroupa', 'blue'), ('Salpeter', 'red')]:
    params = imf_params[name]
    masses = sample_imf(10000, alpha_low=params[0], alpha_high=params[1], mass_break=params[2])
    obs_fractions = []
    for m_lim in mass_limits:
        obs_frac = np.sum(masses > m_lim) / len(masses)
        obs_fractions.append(obs_frac)
    ax.semilogx(mass_limits, obs_fractions, 'o-', color=color, label=name)

ax.set_xlabel('Mass Detection Limit (M☉)')
ax.set_ylabel('Observable Fraction')
ax.set_title('Observable Stars vs Detection Limit')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Initial Mass Function Priors', fontsize=16, fontweight='bold')
save_figure(fig, 'imf_priors')
plt.show()

print("\n✓ IMF prior demonstrations complete")
print("  Key points:")
print("  • Kroupa IMF: broken power law with 3 segments")
print("  • Salpeter IMF: single power law, fewer low-mass stars")
print("  • Choice affects M/L ratios and observable fractions")

## Section 2: Galactic Structure - 3D Density Models

The Galaxy has distinct structural components with different spatial distributions, ages, and metallicities.

### Components

- **Thin Disk**: Scale height ~300 pc, young/metal-rich stars
- **Thick Disk**: Scale height ~900 pc, old/metal-poor stars  
- **Halo**: Power-law profile, very old/metal-poor stars

Each component dominates at different Galactic latitudes and distances.

In [ ]:
from brutus.priors.galactic import logp_galactic_structure, logn_disk, logn_halo

# Create Galactic structure visualization  
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

# Define sightlines
sightlines = [
    (0, 90, 'North Galactic Pole'),
    (0, 0, 'Galactic Center'),
    (180, 0, 'Galactic Anti-center'),
    (90, 0, 'Perpendicular in-plane'),
    (45, 45, 'Intermediate'),
    (270, -30, 'South intermediate')
]

# Distance grid
distances = np.logspace(-2, 2, 200)  # 0.01 to 100 kpc

print("Computing Galactic priors for different sightlines...")

for idx, (l, b, title) in enumerate(sightlines[:6]):
    ax = axes[idx // 3, idx % 3]
    
    coord = np.array([l, b])
    
    # Get prior
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        lnp = logp_galactic_structure(distances, coord)
    
    # Convert to cylindrical coordinates for component visualization
    from astropy.coordinates import SkyCoord, CylindricalRepresentation as CylRep
    import astropy.units as units
    
    ell = np.full_like(distances, l)
    b_arr = np.full_like(distances, b)
    coords = SkyCoord(l=ell * units.deg, b=b_arr * units.deg,
                     distance=distances * units.kpc, frame='galactic')
    coords_cyl = coords.galactocentric.cartesian.represent_as(CylRep)
    R, Z = coords_cyl.rho.value, coords_cyl.z.value
    
    # Compute individual components
    vol_factor = 2 * np.log(distances + 1e-300)
    lnp_thin = logn_disk(R, Z, R_scale=2.6, Z_scale=0.3) + vol_factor
    lnp_thick = logn_disk(R, Z, R_scale=2.0, Z_scale=0.9) + vol_factor + np.log(0.04)
    lnp_halo = logn_halo(R, Z) + vol_factor + np.log(0.005)
    
    # Convert to probabilities
    p_total = np.exp(lnp - np.max(lnp))
    p_thin = np.exp(lnp_thin - np.max(lnp))
    p_thick = np.exp(lnp_thick - np.max(lnp))
    p_halo = np.exp(lnp_halo - np.max(lnp))
    
    # Plot components
    ax.fill_between(distances, p_total, alpha=0.3, color='black', label='Total')
    ax.loglog(distances, p_thin, 'b-', lw=2, alpha=0.7, label='Thin Disk')
    ax.loglog(distances, p_thick, 'g-', lw=2, alpha=0.7, label='Thick Disk')
    ax.loglog(distances, p_halo, 'r-', lw=2, alpha=0.7, label='Halo')
    
    ax.set_xlabel('Distance (kpc)')
    ax.set_ylabel('Relative Probability')
    ax.set_title(f'{title}\n(l={l}°, b={b}°)')
    ax.set_xlim(0.01, 100)
    ax.set_ylim(1e-6, 2)
    ax.grid(True, alpha=0.3)
    
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')

# Panel 7: Metallicity distributions
ax = axes[2, 0]

feh_range = np.linspace(-3, 0.5, 200)

# Component metallicity distributions
thin_feh = np.exp(-(feh_range + 0.1)**2 / (2 * 0.2**2))
thick_feh = np.exp(-(feh_range + 0.6)**2 / (2 * 0.3**2))
halo_feh = np.exp(-(feh_range + 1.5)**2 / (2 * 0.5**2))

ax.plot(feh_range, thin_feh/thin_feh.max(), 'b-', lw=2, label='Thin Disk')
ax.plot(feh_range, thick_feh/thick_feh.max(), 'g-', lw=2, label='Thick Disk')
ax.plot(feh_range, halo_feh/halo_feh.max(), 'r-', lw=2, label='Halo')

ax.set_xlabel('[Fe/H]')
ax.set_ylabel('Relative Probability')
ax.set_title('Metallicity Distributions')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 8: Age distributions
ax = axes[2, 1]

age_range = np.linspace(0, 14, 200)  # Gyr

thin_age = np.exp(-(age_range - 4)**2 / (2 * 3**2)) * (age_range < 10)
thick_age = np.exp(-(age_range - 10)**2 / (2 * 1.5**2)) * (age_range > 8)
halo_age = np.exp(-(age_range - 12)**2 / (2 * 1**2)) * (age_range > 10)

ax.plot(age_range, thin_age/thin_age.max(), 'b-', lw=2, label='Thin Disk')
ax.plot(age_range, thick_age/thick_age.max(), 'g-', lw=2, label='Thick Disk')
ax.plot(age_range, halo_age/halo_age.max(), 'r-', lw=2, label='Halo')

ax.set_xlabel('Age (Gyr)')
ax.set_ylabel('Relative Probability')
ax.set_title('Age Distributions')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 9: Parameters text
ax = axes[2, 2]
ax.axis('off')

components_info = """
Galactic Component Parameters:

Thin Disk:
• Scale height: 300 pc
• Scale length: 2.6 kpc
• Solar density: 0.04 M☉/pc³

Thick Disk:
• Scale height: 900 pc
• Scale length: 3.6 kpc
• Solar density: 0.0025 M☉/pc³

Halo:
• Core radius: 2.0 kpc
• Power law: r^(-3.39)
• Solar density: 0.00015 M☉/pc³
"""

ax.text(0.05, 0.95, components_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Galactic Structure Priors', fontsize=16, fontweight='bold')
save_figure(fig, 'galactic_structure')
plt.show()

print("\n✓ Galactic structure prior demonstrations complete")
print("  Key points:")
print("  • Thin disk dominates at low latitudes")
print("  • Halo becomes important at high latitudes and large distances")
print("  • Each component has distinct [Fe/H] and age distributions")

## Section 3: 3D Dust Extinction - Bayestar Maps

Brutus uses the Bayestar 3D dust maps to estimate extinction as a function of distance and direction.

### Key Concepts

- **3D extinction maps** provide A(V) vs distance
- **Higher extinction** in the Galactic plane
- **Uncertainties increase** with distance
- **Patchy structure** from molecular clouds

In [ ]:
# 3D Dust extinction demonstration
from brutus.priors.extinction import logp_av_bayestar

# Create dust visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: A(V) vs distance for different sightlines
ax = axes[0, 0]

distances = np.logspace(-1, 1.5, 100)  # 0.1 to 30 kpc
sightlines_dust = [
    (0, 0, 'Galactic Center', 'red'),
    (90, 0, 'Perpendicular', 'blue'),
    (180, 0, 'Anti-center', 'green'),
    (45, 45, 'High latitude', 'orange'),
    (0, 90, 'Galactic Pole', 'purple')
]

print("\nComputing dust extinction for different sightlines...")

for l, b, label, color in sightlines_dust:
    coord = np.array([l, b])
    
    # For demonstration, use a simple model
    # Real implementation would use Bayestar
    if abs(b) < 10:  # Near Galactic plane
        av_values = 1.0 * (1 - np.exp(-distances / 2.0)) * np.exp(-abs(b) / 10)
        av_values += 0.1 * np.random.randn(len(distances)) * (distances / 10)
    else:  # High latitude
        av_values = 0.1 * (1 - np.exp(-distances / 5.0)) * np.exp(-abs(b) / 30)
        av_values += 0.02 * np.random.randn(len(distances)) * (distances / 10)
    
    av_values = np.maximum(0, av_values)
    
    ax.plot(distances, av_values, color=color, lw=2, alpha=0.7, label=label)

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('Extinction vs Distance')
ax.set_xscale('log')
ax.set_xlim(0.1, 30)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 2: Extinction uncertainty
ax = axes[0, 1]

distances = np.logspace(-1, 1.5, 50)
av_mean = 0.5 * (1 - np.exp(-distances / 3.0))
av_std = 0.1 * av_mean * (distances / 5.0)

ax.fill_between(distances, av_mean - av_std, av_mean + av_std, 
                alpha=0.3, color='blue', label='1σ uncertainty')
ax.plot(distances, av_mean, 'b-', lw=2, label='Mean A(V)')

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('Extinction Uncertainty Growth')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: A(V) distribution at fixed distance
ax = axes[0, 2]

# Simulate A(V) distribution at 1 kpc
av_samples = np.random.gamma(2, 0.2, 10000)
av_samples = av_samples[av_samples < 5]

ax.hist(av_samples, bins=50, density=True, alpha=0.7, color='green')
ax.axvline(np.median(av_samples), color='red', ls='--', lw=2, label=f'Median = {np.median(av_samples):.2f}')
ax.axvline(np.mean(av_samples), color='blue', ls='--', lw=2, label=f'Mean = {np.mean(av_samples):.2f}')

ax.set_xlabel('A(V) (mag)')
ax.set_ylabel('Probability Density')
ax.set_title('A(V) Distribution at 1 kpc')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: R(V) variation
ax = axes[1, 0]

rv_values = np.linspace(2.5, 5.5, 100)
rv_prob = np.exp(-(rv_values - 3.1)**2 / (2 * 0.3**2))
ax.plot(rv_values, rv_prob / rv_prob.max(), 'b-', lw=2)
ax.axvline(3.1, color='red', ls='--', lw=2, label='Standard R(V) = 3.1')
ax.fill_between(rv_values, rv_prob / rv_prob.max(), alpha=0.3, color='blue')

ax.set_xlabel('R(V)')
ax.set_ylabel('Relative Probability')
ax.set_title('R(V) Variation')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 5: Extinction curve shapes
ax = axes[1, 1]

wavelengths = np.linspace(0.3, 2.5, 200)  # microns
lambda_inv = 1.0 / wavelengths

for rv, color, label in [(2.5, 'blue', 'Dense clouds'), 
                         (3.1, 'black', 'Standard'),
                         (5.0, 'red', 'Diffuse ISM')]:
    # Simplified CCM89 curve
    a_lambda = 1.0 + 0.17699 * (lambda_inv - 1.82) - 0.50447 * (lambda_inv - 1.82)**2
    a_lambda = a_lambda / rv
    ax.plot(wavelengths, a_lambda, lw=2, color=color, label=f'{label} (R_V={rv})')

ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('A(λ) / A(V)')
ax.set_title('Extinction Curve Shapes')
ax.set_xlim(0.3, 2.5)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 6: Dust prior text
ax = axes[1, 2]
ax.axis('off')

dust_info = """
3D Dust Extinction Properties:

Bayestar Map:
• Resolution: ~7 arcmin (HEALPix)
• Distance bins: 31 (0.06-60 kpc)
• Provides E(B-V) estimates
• Based on Pan-STARRS + 2MASS

Key Features:
• Higher extinction in plane
• Patchy structure from clouds
• Uncertainty grows with distance
• R(V) = 3.1 (standard)

Usage in brutus:
• Prior on A(V) given (l,b,d)
• Reddening vector calculation
• Marginalization in fitting
"""

ax.text(0.05, 0.95, dust_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('3D Dust Extinction', fontsize=16, fontweight='bold')
save_figure(fig, 'dust_extinction')
plt.show()

print("\n✓ 3D dust extinction demonstrations complete")
print("  Key points:")
print("  • Extinction increases with distance")
print("  • Higher extinction near Galactic plane")
print("  • Uncertainties grow with distance")

## Section 4: Parallax and Distance Priors

Gaia provides precise parallax measurements that constrain distances, but the parallax-distance transformation is non-linear.

### Key Concepts

- **Non-linear transformation**: d = 1/π (in kpc for π in mas)
- **Asymmetric uncertainties** in distance
- **Lutz-Kelker bias** at low S/N
- **Systematic corrections** for Gaia

In [ ]:
from brutus.priors.astrometric import logp_parallax

# Create parallax/distance visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Parallax-distance transformation
ax = axes[0, 0]

parallaxes = np.linspace(0.1, 10, 1000)  # mas
distances = 1.0 / parallaxes  # kpc

ax.plot(parallaxes, distances, 'b-', lw=2)
ax.fill_between(parallaxes, distances * 0.9, distances * 1.1, 
                alpha=0.3, color='blue', label='±10% uncertainty')

ax.set_xlabel('Parallax (mas)')
ax.set_ylabel('Distance (kpc)')
ax.set_title('Parallax-Distance Relation')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.legend()
ax.grid(True, alpha=0.3)

# Add reference lines
for pi, d in [(10, 0.1), (1, 1), (0.1, 10)]:
    ax.plot(pi, d, 'ro', markersize=8)
    ax.annotate(f'{pi} mas\n{d} kpc', (pi, d), 
               xytext=(5, 5), textcoords='offset points', fontsize=8)

# Panel 2: Distance uncertainty propagation
ax = axes[0, 1]

true_distance = 1.0  # kpc
true_parallax = 1.0  # mas
parallax_errors = [0.01, 0.05, 0.1, 0.2]  # mas
colors = ['blue', 'green', 'orange', 'red']

for sigma_pi, color in zip(parallax_errors, colors):
    measured_parallaxes = np.random.normal(true_parallax, sigma_pi, 10000)
    measured_parallaxes = measured_parallaxes[measured_parallaxes > 0]  # Remove negative
    measured_distances = 1.0 / measured_parallaxes
    
    # Clip extreme values for visualization
    measured_distances = measured_distances[measured_distances < 5]
    
    ax.hist(measured_distances, bins=50, alpha=0.5, density=True,
           label=f'σ_π = {sigma_pi} mas', color=color, histtype='step', lw=2)

ax.axvline(true_distance, color='black', ls='--', lw=2, label='True distance')
ax.set_xlabel('Measured Distance (kpc)')
ax.set_ylabel('Probability Density')
ax.set_title('Distance Uncertainty from Parallax Error')
ax.set_xlim(0, 3)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Lutz-Kelker bias
ax = axes[0, 2]

snr_values = np.linspace(1, 20, 100)
bias_factor = 1 + 1.0 / (2 * snr_values**2)  # Simplified LK bias

ax.plot(snr_values, bias_factor, 'b-', lw=2)
ax.axhline(1.0, color='red', ls='--', label='No bias')
ax.fill_between(snr_values, 1.0, bias_factor, alpha=0.3, color='blue')

ax.set_xlabel('Parallax S/N (π/σ_π)')
ax.set_ylabel('Distance Bias Factor')
ax.set_title('Lutz-Kelker Bias')
ax.set_xlim(1, 20)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Parallax prior given distance
ax = axes[1, 0]

distances = [0.5, 1.0, 2.0, 5.0]  # kpc
parallax_grid = np.linspace(0.01, 5, 500)

for d, color in zip(distances, colors):
    true_pi = 1.0 / d
    sigma_pi = 0.02  # mas (typical Gaia error)
    
    # Gaussian likelihood around true parallax
    lnp = -0.5 * ((parallax_grid - true_pi) / sigma_pi)**2
    p = np.exp(lnp - np.max(lnp))
    
    ax.plot(parallax_grid, p, color=color, lw=2, label=f'd = {d} kpc')

ax.set_xlabel('Observed Parallax (mas)')
ax.set_ylabel('Relative Probability')
ax.set_title('Parallax Likelihood for Different Distances')
ax.set_xlim(0, 3)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 5: Gaia systematic corrections
ax = axes[1, 1]

g_mag = np.linspace(6, 21, 100)
zero_point = -0.03 + 0.001 * (g_mag - 10)  # Simplified zeropoint

ax.plot(g_mag, zero_point * 1000, 'b-', lw=2)  # Convert to μas
ax.axhline(0, color='red', ls='--', label='No correction')
ax.fill_between(g_mag, zero_point * 1000 - 10, zero_point * 1000 + 10,
                alpha=0.3, color='blue', label='±10 μas uncertainty')

ax.set_xlabel('G magnitude')
ax.set_ylabel('Parallax Zeropoint (μas)')
ax.set_title('Gaia DR3 Systematic Corrections')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 6: Prior summary
ax = axes[1, 2]
ax.axis('off')

parallax_info = """
Parallax & Distance Priors:

Key Relations:
• d = 1/π (kpc for π in mas)
• σ_d/d ≈ σ_π/π (small errors)
• Asymmetric for large errors

Lutz-Kelker Bias:
• Affects low S/N measurements
• Pushes distances outward
• Important for π/σ_π < 5

Gaia Systematics:
• Zeropoint corrections (~30 μas)
• Magnitude-dependent
• Color-dependent
• Must be applied!

In brutus:
• Full non-linear transformation
• Proper error propagation
• Systematic corrections included
"""

ax.text(0.05, 0.95, parallax_info, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Parallax and Distance Priors', fontsize=16, fontweight='bold')
save_figure(fig, 'parallax_distance')
plt.show()

print("\n✓ Parallax and distance prior demonstrations complete")
print("  Key points:")
print("  • Non-linear parallax-distance transformation")
print("  • Lutz-Kelker bias at low S/N")
print("  • Gaia systematic corrections essential")

## Section 5: Prior Factorization and Combination

The complete prior in brutus combines all components multiplicatively, with proper factorization based on conditional independence.

### Prior Factorization

The full prior can be written as:

P(θ) = P(M) × P(d,Z,τ|l,b) × P(A_V|d,l,b) × P(π_obs|d)

Where:
- P(M): IMF prior on stellar mass
- P(d,Z,τ|l,b): Galactic structure prior (distance, metallicity, age given position)
- P(A_V|d,l,b): 3D dust prior (extinction given distance and position)
- P(π_obs|d): Parallax likelihood (observed parallax given true distance)

In [ ]:
# Demonstrate prior combination
from brutus.priors import compute_total_lnprior

# Create combined prior visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Prior components along a sightline
ax = axes[0, 0]

distances = np.logspace(-1, 1.5, 200)
l, b = 90, 30  # Intermediate latitude
coord = np.array([l, b])

# Compute individual prior components (simplified)
# Galactic structure
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lnp_gal = logp_galactic_structure(distances, coord)

# Volume factor
lnp_vol = 2 * np.log(distances)

# Dust (simplified)
av_mean = 0.3 * (1 - np.exp(-distances / 3.0))
lnp_dust = -0.5 * (av_mean / 0.2)**2  # Gaussian prior

# Parallax (assuming 1 mas error)
obs_parallax = 0.5  # mas
sigma_parallax = 0.02  # mas
true_parallax = 1.0 / distances
lnp_parallax = -0.5 * ((obs_parallax - true_parallax) / sigma_parallax)**2

# Total prior
lnp_total = lnp_gal + lnp_dust + lnp_parallax

# Convert to probabilities for plotting
p_gal = np.exp(lnp_gal - np.max(lnp_gal))
p_dust = np.exp(lnp_dust - np.max(lnp_dust))
p_parallax = np.exp(lnp_parallax - np.max(lnp_parallax))
p_total = np.exp(lnp_total - np.max(lnp_total))

ax.plot(distances, p_gal, 'b-', lw=2, alpha=0.7, label='Galactic')
ax.plot(distances, p_dust, 'g-', lw=2, alpha=0.7, label='Dust')
ax.plot(distances, p_parallax, 'r-', lw=2, alpha=0.7, label='Parallax')
ax.plot(distances, p_total, 'k-', lw=3, label='Total')

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('Relative Probability')
ax.set_title(f'Prior Components (l={l}°, b={b}°, π={obs_parallax} mas)')
ax.set_xscale('log')
ax.set_xlim(0.1, 30)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: 2D prior surface (distance vs extinction)
ax = axes[0, 1]

d_grid = np.logspace(-0.5, 1, 50)
av_grid = np.linspace(0, 3, 50)
D, AV = np.meshgrid(d_grid, av_grid)

# Compute 2D prior
lnp_2d = np.zeros_like(D)
for i in range(len(d_grid)):
    for j in range(len(av_grid)):
        # Galactic prior at this distance
        lnp_gal_ij = -0.5 * ((D[j, i] - 1.0) / 2.0)**2
        
        # Dust prior
        av_expected = 0.3 * (1 - np.exp(-D[j, i] / 3.0))
        lnp_dust_ij = -0.5 * ((AV[j, i] - av_expected) / 0.2)**2
        
        lnp_2d[j, i] = lnp_gal_ij + lnp_dust_ij

# Plot as contour
p_2d = np.exp(lnp_2d - np.max(lnp_2d))
contours = ax.contourf(D, AV, p_2d, levels=20, cmap='viridis')
ax.contour(D, AV, p_2d, levels=10, colors='white', alpha=0.3, linewidths=0.5)

ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('2D Prior: Distance vs Extinction')
ax.set_xscale('log')
plt.colorbar(contours, ax=ax, label='Relative Probability')

# Panel 3: IMF vs metallicity correlation
ax = axes[1, 0]

masses = np.logspace(-1, 1.5, 100)
feh_values = [-2, -1, 0, 0.3]
colors = ['purple', 'blue', 'green', 'red']

for feh, color in zip(feh_values, colors):
    # IMF prior
    lnp_imf = logp_imf(masses, alpha_low=1.3, alpha_high=2.3, mass_break=0.5)
    
    # Metallicity-dependent correction (hypothetical)
    if feh < -1:  # Metal-poor: fewer low-mass stars
        correction = np.where(masses < 0.5, 0.5, 1.0)
    else:
        correction = 1.0
    
    p_imf = np.exp(lnp_imf - np.max(lnp_imf)) * correction
    ax.loglog(masses, p_imf / p_imf.max(), color=color, lw=2, label=f'[Fe/H] = {feh}')

ax.set_xlabel('Initial Mass (M☉)')
ax.set_ylabel('Relative Probability')
ax.set_title('IMF with Metallicity Dependence')
ax.set_xlim(0.1, 30)
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Prior combination summary
ax = axes[1, 1]
ax.axis('off')

combination_text = """
Prior Factorization in brutus:

Full Prior:
P(θ) = P(M) × P(d,Z,τ|l,b) × P(A_V|d,l,b) × P(π|d)

Components:
1. P(M): IMF prior
   - Kroupa/Salpeter/custom
   - Mass range: 0.08-150 M☉

2. P(d,Z,τ|l,b): Galactic structure
   - Thin disk, thick disk, halo
   - Position-dependent
   - Age & metallicity distributions

3. P(A_V|d,l,b): 3D dust
   - Bayestar maps
   - Distance-dependent
   - Uncertainty propagation

4. P(π|d): Parallax constraint
   - Gaia measurements
   - Non-linear transformation
   - Systematic corrections

Key Properties:
• Multiplicative combination
• Log-space computation
• Proper normalization
• Conditional independence
"""

ax.text(0.05, 0.95, combination_text, transform=ax.transAxes,
       fontsize=9, va='top', family='monospace')

plt.suptitle('Prior Combination and Factorization', fontsize=16, fontweight='bold')
save_figure(fig, 'prior_combination')
plt.show()

print("\n✓ Prior combination demonstrations complete")
print("  Key points:")
print("  • Priors combine multiplicatively")
print("  • Each component provides independent constraints")
print("  • Total prior shapes posterior distribution")

## Summary and Key Takeaways

This tutorial has covered the prior probability distributions used in brutus:

### Key Priors

1. **IMF**: Determines stellar mass distribution
   - Kroupa: Broken power law (standard choice)
   - Salpeter: Single power law
   - Affects M/L ratios and observable fractions

2. **Galactic Structure**: 3D spatial distribution
   - Thin disk: Young, metal-rich, low scale height
   - Thick disk: Old, metal-poor, high scale height  
   - Halo: Very old, very metal-poor, power-law

3. **Dust Maps**: 3D extinction (Bayestar)
   - Provides A(V) as function of distance
   - Higher extinction in Galactic plane
   - Uncertainties increase with distance

4. **Parallax**: Distance constraints from Gaia
   - Non-linear parallax-distance transformation
   - Lutz-Kelker bias pushes distances outward
   - Systematic corrections needed

### Prior Factorization

The full prior combines multiplicatively:

P(θ) = P(M) × P(d,Z,τ|l,b) × P(A_V|d,l,b) × P(π_obs|d)

Components are independent given position and observables.

### Next Steps

- **Tutorial 5**: Fitting Individual Stars with BruteForce
- **Tutorial 6**: Cluster Analysis and Population Fitting
- **Tutorial 7**: 3D Dust Mapping

In [ ]:
print("Tutorial 4 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")